In [8]:
# 모델 매개변수 최적화
# 모델 학습 반복 과정 : 추측 -> 추측과 정답 사이의 손실 계산 -> 손실에 대한 변화율 계산 -> 변화율 기반 최적화

In [9]:
import torch
from torch import nn 
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

training_data = datasets.FashionMNIST(
  root="data",
  train=True,
  download=True,
  transform=ToTensor()
)

test_data = datasets.FashionMNIST(
  root="data",
  train=False,
  download=True,
  transform=ToTensor()
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

class NeuralNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.flatten = nn.Flatten()
    self.linear_relu_stack = nn.Sequential(
      nn.Linear(28 * 28, 512),
      nn.ReLU(),
      nn.Linear(512, 512),
      nn.ReLU(),
      nn.Linear(512, 10),
    )

  def forward(self, x):
    x = self.flatten(x)
    logits = self.linear_relu_stack(x)
    return logits
  
model = NeuralNetwork()

In [10]:
# 하이퍼파라미터: 모델 최적화 과정을 제어할 수 있는 매개변수
# 에폭(epoch) 수 : 데이터셋을 반복하는 횟수
# 배치 크기(batch_size) : 매개변수 갱신 전 신경망을 통해 전파된 데이터 샘플 수
# 학습률(learning rate) : 모델의 매개변수 변화율을 조절하는 보폭(너무 크면 최적값을 지나쳐버리고 너무 작으면 학습이 오래 걸림)
learning_rate = 1e-3
batch_size = 64
epochs = 5

In [11]:
# 손실 함수(loss function) : 획득한 결과와 실제값 사이의 틀린 정도를 측정
# 학습용 데이터 제공 시 학습되지 않은 신경망은 정답을 벗어날 확률이 높기에 학습 중에 이 값을 최소화해야 함(손실 함수가 벗어난 정도를 계산).

# 모델의 출력 logit을 전달하면 logit을 정규화하여 예측 오류 계산
loss_fn = nn.CrossEntropyLoss()

In [12]:
# 최적화(optimizer) : 모델의 오류를 줄이기 위해 모델 매개변수를 조절하는 과정

# 모델 매개변수와 학습률을 전달해 optimizer 초기화
# 인스턴스 생성 순간 가중치와 편향이 랜덤 초기화
# optimizer.zero_grad() : 모델의 변화율을 재설정(기본적으로 변화도는 더해지기 때문에 중복 계산을 막기 위해 재설정)
# loss.backward() : 예측 손실 역전파(손실을 기반으로 변화도 계산 - 각 파라미터의 이동 방향과 그 양을)
# optimizer.step() : 역전파 단계에서 수집된 변화도로 매개변수 조정
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [ ]:
# 전체 구현
def train_loop(dataloader, model, loss_fn, optimizer):
  size = len(dataloader.dataset) # 원본 dataset에 접근
  # 모델을 학습(train) 모드로 설정 - 배치 정규화(Batch Normalization) 및 드롭아웃(Dropout) 레이어들에 중요
  # 이 예시에서는 없어도 되지만, 모범 사례를 위해 추가
  model.train()
  for batch, (X, y) in enumerate(dataloader): # X는 이미지 y는 정답
    # 예측과 손실 계산
    pred = model(X)
    loss = loss_fn(pred, y)

    # 역전파
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if batch % 100 == 0:
      loss, current = loss.item(), batch * batch_size + len(X)
      print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

def test_loop(dataloader, model, loss_fn):
  # 모델을 평가(eval) 모드로 설정합니다 - 배치 정규화(Batch Normalization) 및 드롭아웃(Dropout) 레이어들에 중요합니다.
  # 이 예시에서는 없어도 되지만, 모범 사례를 위해 추가해두었습니다.
  model.eval()
  size = len(dataloader.dataset)
  num_batches = len(dataloader)
  test_loss, correct = 0, 0

  # torch.no_grad()를 사용하여 테스트 시 변화도(gradient)를 계산하지 않도록 함.
  # 이는 requires_grad=True로 설정된 텐서들의 불필요한 변화도 연산 및 메모리 사용량 또한 줆.
  with torch.no_grad():
    for X, y in dataloader:
      pred = model(X)
      test_loss += loss_fn(pred, y).item()
      correct += (pred.argmax(1) == y).type(torch.float).sum().item()

  test_loss /= num_batches
  correct /= size
  print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [ ]:
epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.167997  [   64/60000]
loss: 2.159641  [ 6464/60000]
loss: 2.095455  [12864/60000]
loss: 2.113556  [19264/60000]
loss: 2.069114  [25664/60000]
loss: 2.007850  [32064/60000]
loss: 2.036142  [38464/60000]
loss: 1.955705  [44864/60000]
loss: 1.962509  [51264/60000]
loss: 1.889630  [57664/60000]
Test Error: 
 Accuracy: 50.4%, Avg loss: 1.883243 

Epoch 2
-------------------------------
loss: 1.915821  [   64/60000]
loss: 1.889894  [ 6464/60000]
loss: 1.758465  [12864/60000]
loss: 1.807545  [19264/60000]
loss: 1.709457  [25664/60000]
loss: 1.650127  [32064/60000]
loss: 1.681759  [38464/60000]
loss: 1.575620  [44864/60000]
loss: 1.604683  [51264/60000]
loss: 1.504609  [57664/60000]
Test Error: 
 Accuracy: 60.1%, Avg loss: 1.512873 

Epoch 3
-------------------------------
loss: 1.573177  [   64/60000]
loss: 1.545822  [ 6464/60000]
loss: 1.384973  [12864/60000]
loss: 1.472452  [19264/60000]
loss: 1.363191  [25664/60000]
loss: 1.342153  [32064/600